# Project 2B: POE and IK

This notebook introduces POE-based toolframe kinematics and Jacobian-based inverse kinematics for the Spot leg.


## Environment Setup

To set up the required conda environment for this project, follow these steps:

1. Open a terminal and navigate to the project directory.
2. Run the following command to create the environment:
   ```bash
   conda env create -f 2b_environment.yml
   ```
3. Activate the environment:
   ```bash
   conda activate amr_p2
   ```

Ensure the environment is activated before running any code or simulations.

## Part 1: Kinematics

### Deliverables
- `controllers/Spot_Leg_Kinematics/brain.py`

### Description
In this part of the assignment, you will implement forward and inverse kinematics using the POE (Product of Exponentials). We've separated and fixed one leg of the Spot robot. Your final goal is to make the tool (end-effector) reach the transparent cube on the scene. If implemented correctly, you should be able to control the leg by moving the target cube around in real time. 

### Product of Exponentials (body form)

We model the toolframe pose with joint vector $\theta=[\theta_1,\ldots,\theta_n]^T$ using a body-frame POE expression.
\begin{equation}
T(\theta) = M e^{[\mathcal{B}_1]\theta_1}\cdots e^{[\mathcal{B}_{n-1}]\theta_{n-1}} e^{[\mathcal{B}_n]\theta_n}.
\end{equation}

Here, the home toolframe pose $M=T(0)$ is the toolframe pose when all joint angles are zero, and the body screw axis $\mathcal{B}_i\in\mathbb{R}^6$ is expressed in the home toolframe coordinates.

For a body twist $\mathcal{B}_i=[\omega_i^T, v_i^T]^T$, the corresponding matrix in $\mathfrak{se}(3)$ is the hat map $[\mathcal{B}_i]$.
\begin{equation}
[\mathcal{B}_i] = \begin{bmatrix} [\omega_i] & v_i \\ 0^T & 0 \end{bmatrix}.
\end{equation}

At $\theta=0$, each exponential is identity, so the toolframe pose is exactly the home pose $T(0)=M$.

We've already extracted the $M$ and $B$ matrices for you. They can be found in `controllers/Spot_Leg_Kinematics/robot.py`

For local IK updates, the body velocity twist $\mathcal{V}_b$ is approximated as the body Jacobian $J_b(\theta)$ times a joint increment $\Delta\theta$
\begin{equation}
\mathcal{V}_b \approx J_b(\theta)\,\Delta\theta.
\end{equation}

Deriving the body Jacobian $J_b(\theta)$ from the POE model is the key step before implementing iterative IK.

### GTSAM setup and Pose3 exponential map

For this assignment, the twist vector $\xi$ passed to `gtsam.Pose3.Expmap` uses angular and linear components in one 6D vector.
\begin{equation}
\xi = [\omega_x,\omega_y,\omega_z, v_x, v_y, v_z]^T.
\end{equation}

The POE chain composes home pose $M$ with one exponential map per joint screw axis $\mathcal{B}_i$ and joint angle $\theta_i$.
\begin{equation}
T(\theta) = M \circ \mathrm{Exp}(\mathcal{B}_1\theta_1) \circ \cdots \circ \mathrm{Exp}(\mathcal{B}_n\theta_n).
\end{equation}

### Question 1 (Forward Kinematics)

Go to `controllers/Spot_Leg_Kinematics/brain.py` and implement a function `forward_kinematics`. Use the POE equation described above. Note, that $M$ and $B_{list}$ are already imported from `robot.py`. 

You will need to use GTSAM for exponential map. Here is a reminder on the usage of GTSAM:

In [1]:
import numpy as np
import gtsam

# Example 1: one exponential-map increment
xi = np.array([0.0, 0.0, 0.2, 0.05, 0.0, 0.0])
dT = gtsam.Pose3.Expmap(xi)

# Example 2: update a pose with a body-frame increment
T = gtsam.Pose3()
T_next = T.compose(dT)

### Question 2 (Body Jacobian)

In this question we ask you to analytically derive and then implement the body Jacobian.

The body Jacobian $J_b(\theta)$ for the 3-joint Spot leg toolframe stacks one body twist column per joint.
\begin{equation}
J_b(\theta) = \left[ J_{b,1}(\theta), J_{b,2}(\theta), J_{b,3}(\theta) \right].
\end{equation}

Your derivation should: 
1. Start from the perturbed pose $T(\theta+\delta\theta)$.
2. Keep only first-order terms in $\delta\theta$.
3. Identify each Jacobian column by matching twist terms.

A first-order perturbation of an exponential map keeps only the linear term in the small scalar $\epsilon$.
\begin{equation}
e^{\hat{\xi}\,\epsilon} \approx I + \hat{\xi}\,\epsilon.
\end{equation}

After your derivation, go to `controllers/Spot_Leg_Kinematics/brain.py` and implement a function `jacobian_body` that returns a `6 x n` NumPy array.

Hint: you might want to use `AdjointMap` method from GTSAM.

### Question 3 (Inverse Kinematics via Gradient Descent)

Finally, we ask you to implement an inverse kinematics using gradient descent.

Assume your body Jacobian $J_b(\theta)$ is already available from Question 2.

The body-frame error twist $e_k$ measures the transform from current toolframe pose $T(\theta_k)$ to desired pose $T_d$.
\begin{equation}
e_k = \mathrm{Log}\!\left(T(\theta_k)^{-1}T_d\right) \in \mathbb{R}^6.
\end{equation}

A weighted quadratic objective penalizes rotational and translational components of the error twist $e_k$ through weight matrix $W$.
\begin{equation}
f(\theta_k) = \frac{1}{2} e_k^T W e_k.
\end{equation}

For a small joint update $\Delta\theta$, the error twist linearization uses the body Jacobian $J_b(\theta_k)$.
\begin{equation}
e(\theta_k + \Delta\theta) \approx e_k - J_b(\theta_k)\,\Delta\theta.
\end{equation}

This linearization gives the gradient direction of the objective with respect to joint angles $\theta_k$.
\begin{equation}
\nabla_\theta f(\theta_k) \approx -J_b(\theta_k)^T W e_k.
\end{equation}

A simple gradient-descent step updates the joint vector $\theta_k$ by moving along negative gradient with step size $\alpha>0$.
\begin{equation}
\theta_{k+1} = \theta_k + \alpha J_b(\theta_k)^T W e_k.
\end{equation}

Go to `controllers/Spot_Leg_Kinematics/brain.py` and implement a function `inverse_kinematics`

Suggested algorithm sketch:
1. Initialize joint vector $\theta_0$.
2. Compute current toolframe pose $T(\theta_k)$ using your forward kinematics function.
3. Compute body error twist $e_k = \mathrm{Log}(T(\theta_k)^{-1}T_d)$.
4. Evaluate body Jacobian $J_b(\theta_k)$.
5. Apply the gradient step for $\theta_{k+1}$.
6. Stop when error norm $\|e_k\|$ is below tolerance or max iterations is reached.

Hint: our goal is to match a position of the target cube. We do not care about cube's orientation. How does that affect the weight matrix $W$? 

## Tests and Simulation

After implementing required functions, local tests below should pass.

In [2]:
import unittest

# Load the test suite from test_kinematics.py
loader = unittest.TestLoader()
suite = loader.discover(start_dir="../project_2b/controllers/Spot_Leg_Kinematics", pattern="test_kinematics.py")

# Run the tests and display the results
runner = unittest.TextTestRunner()
runner.run(suite)

...
----------------------------------------------------------------------
Ran 3 tests in 0.042s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

Now, you can open the webots. Use `worlds/spot_leg.wbt` world. In the simulation, you should be able to move the target cube and see the end effector tracking it using inverse kinematics. Make sure to take screenshots of your results and submit to `2c_Reflection_Questions.pptx`.

## Filter Theory Moved

All contact-aided InEKF/filter derivation material has been moved to `Project_2b_Filter.ipynb`.
